In [ ]:
import torch
import os
import matplotlib.pyplot as plt
import numpy as np
from sam2.addons import *

from skimage.measure import regionprops_table, regionprops

from PIL import Image

import torch
import torch.nn.functional as F


from scipy.ndimage import distance_transform_edt
# ---------------------------
# Device (MPS preferred)
# ---------------------------
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()
DTYPE  = torch.float32

import pandas as pd

## Load Data

Each dataset is stored as an `*.h5` file with the following structure:

```text
├── image                 # Original image [x, y], dtype=uint8
├── labels                # Instance label map [x, y], dtype=uint32
├── binary                # Binary  map [x, y], dtype=uint8/bool
├── mask                  # Binary mask of the roving region [x, y], dtype=bool
└── instances/            # Group containing per-instance annotations
    ├── <instance_id>/    # e.g., 0001, 0002, ...
    │   ├── id                    # Integer ID
    │   ├── area                  # Segmented area (float)
    │   ├── bbox                  # Bounding box [x0, y0, w, h]
    │   └── segmentation_cropped  # Cropped mask [w, h], dtype=bool

In [ ]:
file_path = "285_00_03_50x.h5"
file_name = os.path.basename(file_path).split(".")[0]
print(f"file_path: {file_path}\nfile_name: {file_name}")

In [ ]:
file_dict = read_h5(file_path)

In [ ]:
# retunr them as numpy arrays
image       = file_dict["image"]
binary      = file_dict["binary"]
labels      = file_dict["labels"]
mask        = file_dict["mask"]

instances   = file_dict["instances"]

## Display Data

In [ ]:
# --- Create figure ---
fig, axs = plt.subplots(3, 1, figsize=(18, 5), constrained_layout=True)

# --- Common base image ---
for ax in axs:
    ax.imshow(image, cmap="gray")
    ax.axis("off")  # removes ticks and frame
    ax.grid(False)

# --- Overlay different results ---
axs[0].imshow(labels, cmap="plasma", alpha=0.5)
axs[0].set_title("Label overlay", fontsize=14)

axs[1].imshow(binary, cmap="plasma", alpha=0.5)
axs[1].set_title("Binary overlay", fontsize=14)

axs[2].imshow(binary * mask, cmap="plasma", alpha=0.5)
axs[2].set_title("Binary × Mask overlay", fontsize=14)

# --- Figure description ---
fig.suptitle("Comparison of Label, Binary, and Mask Overlays", fontsize=16, weight="bold")

## Refine Mask

the relative coarse and user-defined mask can be refined identifying the outter contour of the roving region from the instance/fiber centers by the convex hull algorithm. A padding of one fiber diameter prevents cutting off fibers at the border of the mask.

In [ ]:
# multiply with mask but keep dtype
labels_masked = (labels * mask).astype(labels.dtype)

props = regionprops_table(
    labels_masked,
    properties=('label', 'centroid', 'equivalent_diameter', 'axis_major_length', 'axis_minor_length'),
)

diameters = props["equivalent_diameter"]

avg_diameter = np.mean(diameters)
print(np.array(diameters).min(),np.array(diameters).max())

print(f"Number of regions: {len(props)}")
print(f"Average equivalent diameter: {avg_diameter:.2f} px")
print(f"Scale based on Average equivalent diameter: {7.0/avg_diameter:.3f} µm/px")

df_props = pd.DataFrame(props)
df_props = df_props.sort_values(by="equivalent_diameter", ascending=True).reset_index(drop=True)

label_id = df_props["label"].iloc[2]
mask_id = labels == label_id

# Use regionprops to get its bounding box
prop = regionprops(mask_id.astype(np.uint8))[0]
minr, minc, maxr, maxc = prop.bbox

# Crop region (optionally add padding)
pad = 10
minr = max(0, minr - pad)
minc = max(0, minc - pad)
maxr = min(labels.shape[0], maxr + pad)
maxc = min(labels.shape[1], maxc + pad)

# Crop arrays
crop_labels = labels[minr:maxr, minc:maxc]
crop_mask   = mask_id[minr:maxr, minc:maxc]

# Show the cropped label region
plt.figure(figsize=(5,5))
plt.imshow(crop_labels, cmap="nipy_spectral")
plt.contour(crop_mask, colors="white", linewidths=0.5)
plt.title(f"Label ID {label_id}")
plt.axis("off")
plt.show()


In [ ]:
from scipy.spatial import ConvexHull
from skimage.draw import polygon
from shapely.geometry import Polygon


pad_radius_px = int(80/2)
print(f"Padding radius = {pad_radius_px} px")

# ------------------------------------
# Extract centroids from df_props
x_coords = df_props["centroid-1"].to_numpy()
y_coords = df_props["centroid-0"].to_numpy()
points = np.column_stack((x_coords, y_coords))

# Compute convex hull
hull = ConvexHull(points)
hull_points = points[hull.vertices]

# ------------------------------------
# Create convex-hull mask
mask_refined = np.zeros_like(image, dtype=np.uint8)
rr, cc = polygon(hull_points[:, 1], hull_points[:, 0], shape=image.shape)
mask_refined[rr, cc] = 1

poly = Polygon(hull_points)
poly_buf = poly.buffer(pad_radius_px, cap_style=1, join_style=2)
mask_padded = np.zeros_like(image, dtype=np.uint8)
if not poly_buf.is_empty:
    ext = np.array(poly_buf.exterior.coords)
    rr, cc = polygon(ext[:,1], ext[:,0], shape=image.shape)
mask_padded[rr, cc] = 1

In [ ]:
# ------------------------------------
# Visualization
plt.figure(figsize=(20, 10))
plt.imshow(image, cmap="gray")
plt.imshow(mask_padded, cmap="plasma", alpha=0.4, label="Padded Hull")
plt.plot(points[hull.vertices, 0], points[hull.vertices, 1], "-", lw=2, color = "tab:purple", label="Original Hull")
plt.title("Convex hull with fiber-radius padding")
plt.axis("off")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.imshow(mask_refined,cmap="plasma")
plt.axis("off")
plt.show()

## Extract Patches: mask-aware tiling

We perform a mask-aware tiling of the refined roving mask into overlapping patches. Only patches with a minimum area coverage (e.g., 80%) of the roving region are retained for further analysis. Patches are extracted with a defined overlap (e.g., 50%) to ensure continuity between neighboring patches.

In [ ]:
def _unfold_patches(x2d, patch_size, stride):
    x = torch.as_tensor(x2d, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # [1,1,H,W]
    unfold = torch.nn.Unfold(kernel_size=(patch_size, patch_size), stride=stride)
    cols = unfold(x)  # [1, p*p, L]
    patches = cols.transpose(1, 2).reshape(-1, patch_size, patch_size)  # [L,p,p]
    return patches

def _coords_for_grid(H, W, p, stride):
    ys = range(0, H - p + 1, stride)
    xs = range(0, W - p + 1, stride)
    return [(y, x) for y in ys for x in xs]

def extract_mask_aware_patches(
    image,            # 2D array, float/uint
    mask,             # 2D binary array {0,1}, same shape as image
    patch_size=256,
    stride_min=64,    # dense base stride to form candidates
    stride_max=256,   # desired sparse spacing deep inside mask
    min_cov=0.5,      # keep patches that cover at least this fraction of mask inside their area
    max_patches=None, # optional cap
    boundary_sharpness=4.0,  # controls how quickly we go dense near edges
):
    """
    1) Generate candidate patches on a dense grid (stride_min).
    2) Score by mask coverage fraction.
    3) Greedy select patches with highest coverage and suppress neighbors
       within a variable radius derived from distance to boundary.
    Returns: coords_kept, img_patches, mask_patches, coverage
    """
    H, W = mask.shape
    p = patch_size

    # (A) Prepare dense candidate grid
    # pad so unfolding works neatly (valid only)
    if H < p or W < p:
        raise ValueError("Image smaller than patch size.")

    # Candidates: unfold mask with stride_min
    mask_t = torch.as_tensor(mask.astype(np.float32))
    mask_patches = _unfold_patches(mask_t.numpy(), p, stride_min)  # [N,p,p]
    N = mask_patches.shape[0]
    coords = _coords_for_grid(H, W, p, stride_min)

    # Coverage = fraction of patch area that is mask==1
    total_area = float(p * p)
    valid_area = mask_patches.sum(dim=(1, 2))  # number of mask-pixels in each patch
    coverage = (valid_area / total_area)       # [N]

    # Filter by minimum coverage
    keep_idx = torch.nonzero(coverage >= float(min_cov)).flatten()
    if keep_idx.numel() == 0:
        return [], torch.empty(0, p, p), torch.empty(0, p, p), torch.empty(0)

    mask_patches = mask_patches[keep_idx]
    coverage = coverage[keep_idx]
    coords = [coords[i] for i in keep_idx.tolist()]

    # Sort candidates by coverage (high → low)
    order = torch.argsort(coverage, descending=True)
    mask_patches = mask_patches[order]
    coverage = coverage[order]
    coords = [coords[i] for i in order.tolist()]

    # (B) Variable suppression radius via distance to boundary
    # Compute distance from each pixel to mask boundary (in pixels).
    # High distance => interior => use stride_max; Near boundary => use stride_min.
    # Normalize with a soft function so radius = lerp(stride_min, stride_max).
    edge = (mask.astype(bool) ^ binary_erosion_safe(mask.astype(bool)))
    # Distance inside mask; set outside to 0 so we don't attract sampling outside.
    dist = distance_transform_edt(mask.astype(bool))
    # Soft mapping (0 near edge -> stride_min, big dist -> stride_max)
    dist_norm = np.tanh(dist / max(1.0, p / boundary_sharpness))  # 0..~1
    # suppression radius map (float)
    rad_map = stride_min + (stride_max - stride_min) * dist_norm  # per-pixel

    # (C) Greedy NMS-style selection with variable radius
    kept_coords = []
    taken = np.zeros(len(coords), dtype=bool)

    # Precompute patch centers for distance checks
    centers = np.array([(y + p/2.0, x + p/2.0) for (y, x) in coords], dtype=np.float32)

    # Helper to read local suppression radius at a given center
    def local_radius_at(yc, xc):
        yi = int(np.clip(round(yc), 0, H - 1))
        xi = int(np.clip(round(xc), 0, W - 1))
        return float(rad_map[yi, xi])

    for i in range(len(coords)):
        if taken[i]:
            continue
        y, x = coords[i]
        cy, cx = centers[i]
        r_i = local_radius_at(cy, cx)

        kept_coords.append((y, x))
        taken[i] = True

        # Suppress neighbors within radius r_i (euclidean on centers)
        if i + 1 < len(coords):
            # Compute squared distances to avoid sqrt
            dy = centers[i+1:, 0] - cy
            dx = centers[i+1:, 1] - cx
            d2 = dy*dy + dx*dx
            # Suppression if distance < r_i
            sup_mask = d2 < (r_i * r_i)
            taken[i+1:] = taken[i+1:] | sup_mask

        if max_patches is not None and len(kept_coords) >= max_patches:
            break

    # (D) Gather final patches
    # Extract image patches aligned with stride_min grid and then index by kept_coords.
    img_t = torch.as_tensor(image, dtype=torch.float32)
    img_patches_all = _unfold_patches(img_t.numpy(), p, stride_min)  # [N_all,p,p]
    # Recompute mapping from kept_coords back to dense grid indices:
    # Build dict from coord -> idx among filtered+sorted coords first
    # But we need indices in original dense grid; simpler: rebuild from coords list
    # Create mapping from coord to index in dense grid
    coords_dense = _coords_for_grid(H, W, p, stride_min)
    coord_to_idx = {c: i for i, c in enumerate(coords_dense)}  # dense grid
    # We need the dense indices for the selected coords (which are subset of filtered/ordered)
    dense_indices = [coord_to_idx[c] for c in kept_coords]
    img_patches_kept = img_patches_all[dense_indices]
    mask_patches_kept = _unfold_patches(mask_t.numpy(), p, stride_min)[dense_indices]
    cov_kept = torch.as_tensor([valid_area := mask_patches_kept[k].sum().item() / total_area
                                for k in range(len(dense_indices))], dtype=torch.float32)

    return kept_coords, img_patches_kept, mask_patches_kept, cov_kept
# ---- small helper for binary erosion without skimage version headaches
from scipy.ndimage import binary_erosion as _bin_erosion
def binary_erosion_safe(bw, radius=1):
    # 3x3 structuring element by default (radius=1)
    se = np.ones((2*radius+1, 2*radius+1), dtype=bool)
    return _bin_erosion(bw, structure=se)

In [ ]:
size = 512
stride = 128
labels_masked = labels * mask

coords_kept, patches_label_kept, patches_mask_kept, cov_kept = extract_mask_aware_patches(
    labels_masked,              # 2D array, float/uint
    mask_refined,               # 2D binary array {0,1}, same shape as image
    patch_size=size,
    stride_min=int(stride//2),  # dense base stride to form candidates
    stride_max=stride,          # desired sparse spacing deep inside mask
    min_cov=0.95,                # keep patches that cover at least this fraction of mask inside their area
    max_patches=None,           # optional cap
    boundary_sharpness=4.0,     # controls how quickly we go dense near edges
)

print(f"{len(coords_kept)} patches kept.")

In [ ]:
fig, axs = plt.subplots(1,3,figsize=(3*3,3))

for i in range(3):
    # pick a random patch index
    i_random = np.random.randint(0, len(patches_label_kept))

    axs[i].imshow(patches_label_kept[i_random].cpu().numpy(), cmap="plasma")
    axs[i].set_title(f"Patch {i_random}")
    axs[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def plot_patch_bboxes(image, coords, patch_size, color="lime", lw=1.5, alpha=0.9):
    """
    Draw bounding boxes for kept patches on top of the original image.

    Parameters
    ----------
    image : np.ndarray
        Original image (H, W) or (H, W, 3).
    coords : list[(y, x)]
        Top-left coordinates for each kept patch (from extract/filter step).
    patch_size : int
        Square patch size used when extracting patches.
    color : str
        Edge color of rectangles.
    lw : float
        Line width.
    alpha : float
        Alpha of rectangle edges.
    """
    H, W = image.shape[:2]
    rects = []

    for (y, x) in coords:
        x0 = max(0, min(W, x))
        y0 = max(0, min(H, y))
        x1 = max(0, min(W, x + patch_size))
        y1 = max(0, min(H, y + patch_size))
        if x1 <= x0 or y1 <= y0:   # fully outside after clipping
            continue
        rects.append(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False))

    fig, ax = plt.subplots(figsize=(20, 10))
    if image.ndim == 2:
        ax.imshow(image, cmap="gray")
    else:
        ax.imshow(image)
    if rects:
        pc = PatchCollection(rects, facecolor="none", edgecolor=color, linewidths=lw, alpha=alpha)
        ax.add_collection(pc)
    ax.set_title(f"Kept patches: {len(rects)}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_patch_bboxes(image, coords_kept, patch_size=size, color="tab:purple", lw=1.5)

## Spatial Estimation of the statistical RVE size

In [ ]:
@torch.no_grad()
def s2_descriptor(patches: torch.Tensor, radial: bool = False, eps: float = 1e-12):
    """
    Compute two-point correlation S2 for binary patches.

    Parameters
    ----------
    patches : torch.Tensor
        Shape [B, H, W], values in {0,1} (float/bool is fine). On CPU/GPU/MPS.

    Returns
    -------
    result : dict
        - 'S2':   (B,H,W)      centered two-point correlation
    """
    assert patches.ndim == 3, "Expected [B,H,W]"
    x = patches.to(torch.float32)

    B, H, W = x.shape


    # ---- S2 via FFT (autocorrelation normalized by N) ----
    F = torch.fft.fft2(x, dim=(-2, -1))
    S2 = torch.real(torch.fft.ifft2(F * torch.conj(F), dim=(-2, -1))) / (H * W)
    S2 = torch.fft.fftshift(S2, dim=(-2, -1))  # center
    
    if radial:
        # Precompute radius bins (shared for batch)
        device = x.device
        y = torch.arange(H, device=device)
        z = torch.arange(W, device=device)
        yy, zz = torch.meshgrid(y, z, indexing="ij")
        cy, cz = (H - 1) / 2.0, (W - 1) / 2.0
        r = torch.sqrt((yy - cy) ** 2 + (zz - cz) ** 2)
        r_int = r.round().to(torch.int64)
        rmax = int(r_int.max().item())

        vals = S2.reshape(B, -1)
        bins = r_int.reshape(-1)
        prof = torch.zeros((B, rmax + 1), device=device, dtype=vals.dtype)
        prof.scatter_add_(1, bins.unsqueeze(0).expand(B, -1), vals)

        # counts per radius
        cnt = torch.zeros(rmax + 1, device=device, dtype=vals.dtype)
        cnt.scatter_add_(0, bins, torch.ones_like(bins, dtype=vals.dtype, device=device))
        prof = prof / torch.clamp(cnt, min=eps)

        return S2, prof  # (B, R)
    
    return S2


@torch.no_grad()
def phi_descriptor(patches: torch.Tensor):
    """
    Compute volume fraction (phi) for binary patches.

    Parameters
    ----------
    patches : torch.Tensor
        Shape [B, H, W], values in {0,1} (float/bool is fine). On CPU/GPU/MPS.
    Returns
    -------
    result : dict
        - 'phi':  (B,)         volume fraction
    """
    assert patches.ndim == 3, "Expected [B,H,W]"
    x = patches.to(torch.float32)

    B, H, W = x.shape

    # ---- phi ----
    phi = x.mean(dim=(1, 2))  # (B,)
    return phi


@torch.no_grad()
def corr_length_halfheight(S2r: torch.Tensor, phi: torch.Tensor) -> torch.Tensor:
    """
    Half-height correlation length per patch, using r = 1 as the peak reference.
    S2r: [B, R], phi: [B] in [0,1]
    Returns: zeta [B] (int64), first r>=1 where C(r) <= 0.5*C(1); R-1 if never drops.
    """
    B, R = S2r.shape
    base = (phi ** 2).unsqueeze(1)      # [B,1]
    C = S2r - base                      # [B,R]

    # Use C at index 1 as reference
    C1 = C[:, 1].clamp_min(1e-12)       # [B]
    thresh = 0.5 * C1                   # [B]

    # Condition only for indices >= 1
    cond = C[:, 1:] <= thresh[:, None]  # [B, R-1]

    # Find first such index (relative to r=1)
    has_hit = cond.any(dim=1)
    idx_rel = torch.argmax(cond.to(torch.int32), dim=1)  # range: 0..R-2

    # Build final absolute indices
    idx = torch.full((B,), R-1, device=S2r.device, dtype=torch.int64)
    idx[has_hit] = idx_rel[has_hit] + 1   # shift back by +1 to real r-index

    return idx

@torch.no_grad()
def integral_range_from_S2r(S2r: torch.Tensor, phi: torch.Tensor) -> torch.Tensor:
    """
    Same as before, batched. Returns A_int [B] in pixel^2.
    """
    B, R = S2r.shape
    base = (phi ** 2).unsqueeze(1)
    C = S2r - base
    Cpos = torch.clamp(C, min=0.0)
    r = torch.arange(R, device=S2r.device, dtype=S2r.dtype)  # [R]
    return (2.0 * torch.pi * (Cpos * r).sum(dim=1))          # [B]

@torch.no_grad()
def rve_size_from_integral_range(phi_mean: torch.Tensor, A_int_mean: torch.Tensor, cv_target: float) -> torch.Tensor:
    return torch.sqrt((phi_mean * (1.0 - phi_mean)) * A_int_mean / (cv_target ** 2))

In [ ]:
patches_tensor = (patches_label_kept > 0).float()  # binary as float

patches_S2, patches_S2r = s2_descriptor(patches_tensor, radial=True)
patches_phi = phi_descriptor(patches_tensor)
patches_corrlen = corr_length_halfheight(patches_S2r, patches_phi)

B, H, W = patches_tensor.shape
R = patches_S2r.shape[1]
r_np = np.arange(R)                 # lag in pixels
size = H                            # assuming square patches


# --- 1) Integral range per patch (in pixel^2) ---
A_int = integral_range_from_S2r(patches_S2r, patches_phi)    # [B]

# --- 2) Averages over all patches ---
phi_mean = patches_phi.mean()         # scalar
A_int_mean = A_int.mean()             # scalar

# --- 3) Choose target COV (e.g. 0.05 for 5% scatter) ---
cv_target = 0.05  # or 0.10 for 10%

# --- 4) RVE side length in pixels ---
L_rve_px = rve_size_from_integral_range(phi_mean, A_int_mean, cv_target)

print(f"phi_mean       = {phi_mean.item():.4f}")
print(f"A_int_mean     = {A_int_mean.item():.2f} px^2")
print(f"Target COV     = {cv_target:.3f}")
print(f"RVE side       = {L_rve_px.item():.1f} px")



fig, axs = plt.subplots(4, 4, figsize=(4*3, 3*3))

for i in range(4):
    i_random = np.random.randint(0, len(patches_tensor))

    phi_i = patches_phi[i_random].cpu().numpy()
    corrlen_i = int(patches_corrlen[i_random].cpu().numpy())

    print(f"Patch {i_random}: phi={phi_i:.4f}, corrlen={corrlen_i} px")

    # Row 0: grayscale patch
    im0 = axs[0][i].imshow(patches_tensor[i_random].cpu().numpy(), cmap="gray")
    axs[0][i].set_title(r"$\phi$=" + f"{phi_i:.4f}")
    fig.colorbar(im0, ax=axs[0][i])

    # Row 1: S2 2D map (plasma)
    im1 = axs[1][i].imshow(patches_S2[i_random].cpu().numpy(), cmap="plasma")
    fig.colorbar(im1, ax=axs[1][i])

    # Row 2: 1D S2(r) vs lag r
    axs[2][i].plot(r_np[1:], patches_S2r[i_random, 1:].cpu().numpy())
    axs[2][i].axvline(corrlen_i, ls="--")  # mark correlation length
    axs[2][i].axvline(L_rve_px.item(), ls="-")  # mark correlation length
    # Row 3: centered C(r) = S2(r) - phi^2 vs lag r
    C_i = patches_S2r[i_random].cpu().numpy() - phi_i**2
    axs[3][i].plot(r_np[1:], C_i[1:])
    axs[3][i].axvline(corrlen_i, ls="--")
    axs[3][i].axvline(L_rve_px.item(), ls="-")  # mark correlation length


# Formatting
for ax in axs[2]:
    ax.set_xlabel(r"$r$ [px]")
    ax.set_ylabel(r"$S_2(r)$")
    ax.grid()
    ax.set_ylim((0.3, 0.8))
    ax.set_xlim((0, size // 2))

for ax in axs[3]:
    ax.set_xlabel(r"$r$ [px]")
    ax.set_ylabel(r"$C(r)$")
    ax.grid()
    ax.set_ylim((-0.1, 0.3))
    ax.set_xlim((0, size // 2))

for ax in axs[0]:
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")

for ax in axs[1]:
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")

fig.tight_layout()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------------------
# Prepare data
# ---------------------------------------------------------------------
# S2_list: e.g. S2 - phi^2
S2_list = [
    (p.cpu().numpy() - phi.cpu().numpy()**2)
    for p, phi in zip(patches_S2[:30], patches_phi[:30])
    ]

# S2_list = [
#     (p.cpu().numpy())
#     for p, phi in zip(patches_S2[:30], patches_phi[:30])
# ]


# Image/phi patches to show in the right subplot (adjust as needed)
img_list = [patch.cpu().numpy() for patch in patches_tensor[:30]]

H, W = S2_list[0].shape
x = np.arange(W)
y = np.arange(H)
X, Y = np.meshgrid(x, y)

# Global color ranges
cmin_s2 = min(Z.min() for Z in S2_list)
cmax_s2 = max(Z.max() for Z in S2_list)

cmin_img = min(im.min() for im in img_list)
cmax_img = max(im.max() for im in img_list)

# ---------------------------------------------------------------------
# Create subplots: left = 3D surface, right = 2D image/heatmap
# ---------------------------------------------------------------------
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "surface"}, {"type": "xy"}]],
    column_widths=[0.6, 0.4],
    horizontal_spacing=0.08,
    subplot_titles=("C₂ surface", "Patch / image"),
)

# ---- Initial traces (frame 0) ----
fig.add_trace(
    go.Surface(
        x=X,
        y=Y,
        z=S2_list[0],
        colorscale="Plasma",
        showscale=True,
        cmin=cmin_s2,
        cmax=cmax_s2,
        name="C₂ surface",
        colorbar=dict(
            title="C₂ [-]",
            x=0.46,        # <<-- Move colorbar BETWEEN the subplots
            thickness=15,
            len=0.8,       # shorten a bit for aesthetics
        ),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Heatmap(
        z=img_list[0],
        colorscale="Gray",
        showscale=True,
        zmin=cmin_img,
        zmax=cmax_img,
        name="Patch",
        colorbar=dict(title="Intensity", x=1.02),  # move slightly right
    ),
    row=1,
    col=2,
)

# ---------------------------------------------------------------------
# Animation frames: each frame updates BOTH traces
# ---------------------------------------------------------------------
frames = []
for k, (Z_s2, Z_img) in enumerate(zip(S2_list, img_list)):
    frames.append(
        go.Frame(
            name=str(k),
            data=[
                go.Surface(
                    x=X,
                    y=Y,
                    z=Z_s2,
                    colorscale="Plasma",
                    showscale=True,
                    cmin=cmin_s2,
                    cmax=cmax_s2,
                    name="C₂ surface",
                    colorbar=dict(
                        title="C₂ [-]",
                        x=0.46,   # <<-- same position during animation
                        thickness=15,
                        len=0.8,
                    ),
                ),
                go.Heatmap(
                    z=Z_img,
                    colorscale="Gray",
                    showscale=True,
                    zmin=cmin_img,
                    zmax=cmax_img,
                    name="Patch",
                ),
            ],
            # layout=go.Layout(title_text=f"S₂ Patch #{k}"),
        )
    )

fig.frames = frames

# ---------------------------------------------------------------------
# Play / Pause buttons + slider
# ---------------------------------------------------------------------
fig.update_layout(
    width=900,
    height=500,
    # title="S₂ Patches (Surface + Image, Animated)",
    scene=dict(
        xaxis_title="X [px]",
        yaxis_title="Y [px]",
        zaxis_title="C₂ [-]",
        aspectmode="cube",
    ),
    xaxis2=dict(
        title="X [px]",
        scaleanchor="y2",
        constrain="domain",
    ),
    yaxis2=dict(
        title="Y [px]",
        autorange="reversed",  # image-like orientation
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            x=0.05,
            y=1.15,
            xanchor="left",
            yanchor="top",
            direction="left",
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=150, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=False),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            currentvalue={"prefix": "Patch: "},
            pad={"t": 30},
            steps=[
                dict(
                    label=str(k),
                    method="animate",
                    args=[
                        [str(k)],
                        dict(
                            frame=dict(duration=0, redraw=True),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                )
                for k in range(len(S2_list))
            ],
        )
    ],
)

fig.show()
fig.write_html(f"{file_name}_s2_patches_animation.html")


In [ ]:
patches_image = []

for i, (coords) in enumerate(coords_kept):
    y, x = coords
    patches_image.append(image[y:y+size, x:x+size])
patches_image = np.stack(patches_image)


print(f"patches_image.shape: {patches_image.shape}")

In [ ]:
import h5py

def to_numpy(x):
    """Converts torch tensors to numpy safely."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

with h5py.File(file_path, "a") as f:
    # ---------------------------------------------------------------------
    # 0. Create main group safely
    # ---------------------------------------------------------------------
    if "analysis" in f:
        del f["analysis"]      # overwrite old analysis block
    grp = f.create_group("analysis")

    # ---------------------------------------------------------------------
    # 1. Save metadata as attributes
    # ---------------------------------------------------------------------
    grp.attrs["patch_size"] = size
    grp.attrs["stride_min"] = int(stride // 2)
    grp.attrs["stride_max"] = stride
    grp.attrs["min_cov"] = 0.9
    grp.attrs["boundary_sharpness"] = 4.0
    grp.attrs["num_patches"] = len(coords_kept)
    grp.attrs["phi_mean"] = float(phi_mean)
    grp.attrs["A_int_mean_px2"] = float(A_int_mean)
    grp.attrs["cv_target"] = float(cv_target)
    grp.attrs["L_rve_px"] = float(L_rve_px)

    # # ---------------------------------------------------------------------
    # # 2. Save coordinates and masks
    # # ---------------------------------------------------------------------
    grp.create_dataset("coords", data=to_numpy(coords_kept), compression="gzip")

    # if "kept_mask" in globals():
    #     grp.create_dataset("mask", data=to_numpy(kept_mask), compression="gzip")
    grp.create_dataset("mask_refined", data=to_numpy(mask_refined), compression="gzip")
    # ---------------------------------------------------------------------
    # 3. Save S₂ patches
    #    patches_S2: list/array of [N, H, W]
    # ---------------------------------------------------------------------
    S2_np = to_numpy(patches_S2)  # shape (N, H, W)
    grp.create_dataset("S2", data=S2_np, compression="gzip")

    # ---------------------------------------------------------------------
    # 4. Save φ patches
    # ---------------------------------------------------------------------
    phi_np = to_numpy(patches_phi)  # shape (N, H, W)
    grp.create_dataset("phi", data=phi_np, compression="gzip")

    # ---------------------------------------------------------------------
    # 5. Save input image data (optional)
    # ---------------------------------------------------------------------

    grp.create_dataset("patches_label", data=to_numpy(patches_label_kept), compression="gzip")
    grp.create_dataset("patches_image", data=to_numpy(patches_image), compression="gzip")

    print(f"Saved analysis results to {file_path} under group 'analysis'.")